# **Goal**

This notebook applies **clustering** techniques to a dataset of Spotify tracks to identify
**sound profile patterns** and group tracks with similar characteristics. The analysis explores
how audio features relate to each other and lays the groundwork for a **music recommendation
system** based on how songs sound, regardless of genre or popularity.

---

# **About the dataset**

[Spotify Tracks Attributes and Popularity](https://www.kaggle.com/datasets/melissamonfared/spotify-tracks-attributes-and-popularity)
(Kaggle) contains ~114k Spotify tracks across 114 genres, with:

* **Audio features:** danceability, energy, loudness, acousticness, instrumentalness, valence, tempo, among others.
* **Popularity:** Spotify's popularity index.
* **Metadata:** track name, artists, album name, genre.

Spotify no longer exposes audio features to new apps (since November 2024), so this dataset is
the only source of audio features for the project.

---

# **Columns**

1. **track_id:** Spotify track ID.
2. **track_name:** Track name.
3. **artists:** Artist(s), separated by `;`.
4. **album_name:** Album the track was released on.
5. **track_genre:** Genre the track was collected under.
6. **popularity:** Spotify popularity index (0 to 100).
7. **duration_ms:** Track length in milliseconds.
8. **explicit:** Whether the track has explicit lyrics.
9. **danceability:** How suitable the track is for dancing (0 to 1).
10. **energy:** Perceived intensity and activity (0 to 1).
11. **key:** Musical key, as a pitch class (0 = C, 1 = C♯/D♭, …, 11 = B).
12. **loudness:** Average loudness (dB).
13. **mode:** Modality (1 = major, 0 = minor).
14. **speechiness:** Presence of spoken words (0 to 1).
15. **acousticness:** Confidence that the track is acoustic (0 to 1).
16. **instrumentalness:** Likelihood that the track has no vocals (0 to 1).
17. **liveness:** Likelihood that the track was recorded live (0 to 1).
18. **valence:** Musical positiveness (0 to 1).
19. **tempo:** Estimated tempo in BPM.
20. **time_signature:** Estimated beats per bar (values 0 to 5 in this dataset).

## **Loading and inspecting the data**

In [ ]:
# Download the dataset from Kaggle (public, no credentials needed)
!pip install -q kagglehub

import kagglehub

# Link: https://www.kaggle.com/datasets/melissamonfared/spotify-tracks-attributes-and-popularity
path = kagglehub.dataset_download('melissamonfared/spotify-tracks-attributes-and-popularity')

import os

# List the downloaded files
for entry in os.listdir(path):
    print(entry)

In [ ]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(path, 'dataset.csv'))

print("First rows of the dataset:")
display(df.head())
print("\nShape:", df.shape)

In [ ]:
print(f"Rows and columns: {df.shape}")

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTop 20 genres by number of tracks:")
print(df['track_genre'].value_counts().head(20))

print("\nPopularity distribution:")
print(df['popularity'].describe())

# Pairwise correlations between numeric columns
corr = df.corr(numeric_only=True)
corr_pairs = (
    corr.unstack()
    .sort_values(ascending=False)
    .drop_duplicates()
)
print("\nTop 10 correlations (absolute value) between numeric columns:")
print(corr_pairs[(corr_pairs < 1.0)].abs().sort_values(ascending=False).head(10))

## **Exploratory analysis: first observations**

**1.** Only one row has missing values (in *artists*, *album_name* and *track_name*). These columns are not used as clustering features.

**2.** Every genre appears exactly 1000 times in the raw data. This balance is partly artificial: the same track can be listed under several genres (see *Removing duplicate tracks* below).

**3.** Popularity is spread out and most tracks have low popularity, so the dataset is not made only of hits.

**4. Main correlations:** some audio features are strongly and consistently correlated:
* energy – loudness (0.76): more energetic tracks tend to be louder.
* energy – acousticness (−0.73): acoustic tracks tend to have low energy.
* acousticness – loudness (−0.59): more acoustic tracks tend to be quieter.
* danceability – valence (0.48): more danceable tracks tend to sound "happier".

*Popularity is **not** strongly correlated with any audio feature. A track's success seems to depend on external factors (marketing, artist, playlists, etc.) rather than on how it sounds.*

## **Data cleaning**

In [ ]:
# Drop rows with missing values (only 1 row in this dataset)
df_clean = df.dropna().copy()

print("Rows removed because of missing values:", len(df) - len(df_clean))

### Removing duplicate tracks

The dataset has two kinds of duplicates:

1. **Same `track_id`, different genre.** The dataset was built by collecting ~1000 tracks
   per genre, so a track tagged with several genres appears once per genre
   (e.g. *I'll Be Home For Christmas* by Kacey Musgraves and Lana Del Rey appears 5 times).
   The audio features are identical; only `track_genre` changes.
2. **Same song (name + artists), different `track_id`.** The same recording released as a
   single and on an album gets a different Spotify ID (e.g. *The Hills* by The Weeknd,
   once with popularity 88 and once with popularity 3).

If kept, duplicates would be recommended as "similar" to themselves and would give
multi-genre tracks extra weight in the clustering.

**Decision:** keep one row per track and collect all its genres into a `genres` list
instead of dropping them, so no genre information is lost (e.g. for filtering by genre).
When the same song has several IDs, keep the most popular version, which is the one
Spotify surfaces as canonical. `track_genre` keeps the genre of the row that was kept,
so the existing analysis below still works.

In [ ]:
# 1) Same track_id listed under several genres: keep one row, collect all genres.
# Sorting by popularity (stable, with track_id as tie-breaker) makes the kept row deterministic.
rows_before = len(df_clean)

genres_by_id = df_clean.groupby('track_id')['track_genre'].agg(lambda s: sorted(set(s)))

df_clean = (
    df_clean
    .sort_values(['popularity', 'track_id'], ascending=[False, True], kind='stable')
    .drop_duplicates('track_id')
)
df_clean['genres'] = df_clean['track_id'].map(genres_by_id)

print(f"Rows removed (duplicate track_id): {rows_before - len(df_clean):,}")
print(f"Tracks left: {len(df_clean):,}")
print(f"Tracks with more than one genre: {(df_clean['genres'].str.len() > 1).sum():,}")

In [ ]:
# 2) Same song (name + artists) under different track_ids: keep the most popular version,
# merging the genres of all versions.
song_key = ['track_name', 'artists']
rows_before = len(df_clean)

print("Example before:")
display(df_clean.loc[df_clean['track_name'] == 'The Hills', ['track_id', 'artists', 'album_name', 'popularity']])

genres_by_song = (
    df_clean.groupby(song_key)['genres']
    .agg(lambda lists: sorted(set().union(*lists)))
)

df_clean = (
    df_clean
    .drop(columns='genres')
    .drop_duplicates(song_key)  # already sorted by popularity, so the most popular version is kept
    .join(genres_by_song, on=song_key)
    .sort_values('index')       # restore the original dataset order
    .reset_index(drop=True)
)

print(f"Rows removed (same name + artists): {rows_before - len(df_clean):,}")
print(f"Tracks left: {len(df_clean):,}")
assert not df_clean.duplicated('track_id').any()
assert not df_clean.duplicated(song_key).any()

In [ ]:
# Dedup side effect: the dataset is no longer balanced by genre.
# Genres whose tracks overlap with others (e.g. pop, rock) lose more rows.
genre_counts = df_clean['track_genre'].value_counts()
print(f"Tracks per genre after dedup: min {genre_counts.min()}, max {genre_counts.max()}")
display(genre_counts.tail(10))

### Choosing the features

Besides the continuous audio features, the dataset has columns that look numeric but are not
good inputs for a distance-based method:

* **`key`** is a pitch class (0 = C … 11 = B). It is categorical and cyclic: key 11 is not
  "bigger" than key 0, and B is next to C. Treating it as a number creates false distances.
* **`mode`** (major/minor) and **`explicit`** are binary. After standardization, two otherwise
  identical tracks that differ only in `mode` end up ~2 units apart, a huge gap compared to
  continuous features.
* **`time_signature`** is almost constant (most tracks are in 4/4) and has invalid values
  (0 and 1 beats per bar).
* **`duration_ms`** describes length, not sound, and has extreme outliers.

The cell below checks whether any of them carries information about the *vibe* of a track.

In [ ]:
candidates = ['key', 'mode', 'time_signature', 'explicit', 'duration_ms']

print("time_signature counts:", df_clean['time_signature'].value_counts().sort_index().to_dict())
print(f"Share of 4/4 tracks: {(df_clean['time_signature'] == 4).mean():.0%}")
print(f"Share of explicit tracks: {df_clean['explicit'].mean():.1%}")
print("Mean valence by mode (0 = minor, 1 = major):",
      df_clean.groupby('mode')['valence'].mean().round(3).to_dict())
print("Duration in minutes (min / median / max):",
      (df_clean['duration_ms'].agg(['min', 'median', 'max']) / 60_000).round(1).tolist())

print("\nCorrelation with the main mood features:")
display(
    df_clean[candidates + ['valence', 'energy', 'danceability']]
    .astype(float)
    .corr()
    .loc[candidates, ['valence', 'energy', 'danceability']]
    .round(2)
)

**Decision:** use only the 9 continuous audio features:
`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`,
`liveness`, `valence` and `tempo`.

* None of the dropped columns is meaningfully related to mood. Notably, minor-key tracks are
  barely less positive than major-key ones (mean valence ~0.46 vs ~0.47), so `mode` does not
  work as a proxy for "sad" in this data.
* These 9 features are also the ones people use to describe music in words ("energetic",
  "acoustic", "melancholic"…), which keeps the similarity interpretable.
  Nobody describes a song as "in D♯ in 5/4".
* The dropped columns stay in the dataset as **metadata**: `explicit`, for example, can be
  used as a filter instead of a similarity feature.

In [ ]:
# Audio features used for clustering and similarity (see "Choosing the features" above).
# IDs, text, genre, popularity and the dropped columns stay in df_clean as metadata.
FEATURES = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
            'instrumentalness', 'liveness', 'valence', 'tempo']
df_features = df_clean[FEATURES]

print("Columns used for clustering:")
print(df_features.columns.tolist())

# Standardize the features (essential for K-Means, which is distance-based)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_features)

print("\nScaled dataset shape:", df_scaled.shape)

df_scaled = pd.DataFrame(df_scaled, columns=df_features.columns)

df_scaled.head()

## **About the cleaning steps**

To prepare the data for K-Means, we applied a few transformations so that only relevant and
comparable information reaches the algorithm:

**1. Missing values:**
Only 1 row had missing values, and it was dropped. Since the remaining data was complete, no imputation was needed.

**2. Duplicate tracks:**
Duplicates by `track_id` and by name + artists were removed, as explained above.

**3. Feature selection:**
Only the 9 continuous audio features are used, as explained in *Choosing the features*.
IDs, text, genre and *popularity* (which could bias the clusters towards "success" instead of
how the tracks sound) are kept as metadata but not used in distance calculations.

**4. Standardization (StandardScaler):**
StandardScaler transforms every feature to:
* mean ≈ 0
* standard deviation ≈ 1

This matters because the features have very different scales: most range from 0 to 1, while
loudness is in dB (roughly −50 to 5) and tempo in BPM (up to ~240). Without standardization,
K-Means would be dominated by tempo and loudness.

## **Exploratory data analysis**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("\nData types:")
print(df.dtypes)

# Descriptive statistics of the numeric columns
num_cols = ['popularity', 'duration_ms', 'danceability', 'energy', 'loudness',
            'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

stats = df[num_cols].describe()
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("\nDescriptive statistics:")
display(stats)

# Popularity distribution
plt.figure(figsize=(8,5))
sns.histplot(df['popularity'], bins=30, kde=True, color='mediumseagreen')
plt.title('Track popularity distribution')
plt.xlabel('Popularity')
plt.ylabel('Count')
plt.show()

# Genre distribution (top 20)
top_genres = df['track_genre'].value_counts().head(20)
plt.figure(figsize=(10,6))
sns.barplot(y=top_genres.index, x=top_genres.values, color='mediumseagreen')
plt.title('Top 20 genres')
plt.xlabel('Number of tracks')
plt.ylabel('Genre')
plt.show()

# Correlation between numeric columns
plt.figure(figsize=(12,10))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation matrix of numeric columns')
plt.show()

## **Choosing the number of clusters (k)**

k has to be chosen *before* training the final model. Two complementary criteria:

* **Elbow method:** inertia (within-cluster sum of squares) always decreases as k grows; we look
  for the point where adding a cluster stops reducing it much.
* **Silhouette score:** from −1 to 1, how much closer each track is to its own cluster than to
  the nearest other cluster. Values near 0 mean clusters overlap.

Each k is fit on the **full** dataset with 10 initializations (`n_init=10`). Silhouette is
O(n²), so it is evaluated on a 20,000-track sample of that full-data clustering.

*Note:* the first version of this notebook trained with k=8 before running this analysis, and
computed silhouette by fitting K-Means on a 10% sample with a single initialization. That setup
was unstable and inflated some scores (k=2 reached ~0.50 on the sample vs ~0.26 here).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = df_scaled
k_values = range(2, 13)

results = []
for k in k_values:
    model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    sil = silhouette_score(X, model.labels_, sample_size=20_000, random_state=42)
    results.append({'k': k, 'inertia': model.inertia_, 'silhouette': sil})

k_scores = pd.DataFrame(results).set_index('k')
# How much inertia each extra cluster removes, relative to k-1
k_scores['inertia_drop_%'] = -k_scores['inertia'].pct_change() * 100
display(k_scores.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_scores.index, k_scores['inertia'], marker='o')
axes[0].set_title('Elbow method')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].grid(True)

axes[1].plot(k_scores.index, k_scores['silhouette'], marker='o')
axes[1].set_title('Silhouette score by number of clusters')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Stability check: are the silhouette scores of the candidate k values robust to the random seed?
for k in (2, 6, 7, 8):
    scores = [
        silhouette_score(
            X,
            KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X),
            sample_size=20_000,
            random_state=seed,
        )
        for seed in (0, 1, 2)
    ]
    print(f"k={k}: silhouette {np.mean(scores):.3f} ± {np.std(scores):.3f}")

**Decision: k = 7.**

* **Elbow:** up to k=7 each extra cluster removes ~9–11% of the inertia; at k=8 the gain drops to
  ~5% and keeps shrinking.
* **Silhouette:** among k ≥ 3, k=7 has the highest score (~0.20), and it is stable across seeds.
* **Why not k=2?** It has the highest silhouette (~0.26), but it only splits the catalog into
  "calm/acoustic" vs "energetic". That is real, but too coarse to describe the music.

**Being honest about the result:** every silhouette score is below 0.26, which means the clusters
are **weak**. Tracks do not form well-separated groups in audio-feature space; they form a
continuum, and many tracks sit near the border between clusters. The clusters are still useful to
*describe* the catalog (see the profiles below), but they should not be used to restrict
recommendations: a track near a border may have its closest neighbours in another cluster.

## **Training K-Means**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Standardized features (one row per track)
X = df_scaled

# Train the final model with the k chosen above
n_clusters = 7
kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
kmeans.fit(X)

# Cluster assigned to each track
clusters = kmeans.labels_

# PCA (Principal Component Analysis) with 6 components
pca = PCA(n_components=6, random_state=42)
X_pca = pca.fit_transform(X)

df_pca = pd.DataFrame(data=X_pca, columns=[f'PC{i+1}' for i in range(6)])
df_pca['cluster'] = clusters

df_pca.head()

### What each cluster represents

Mean of each audio feature (original scale) per cluster, plus the most common genres.

In [ ]:
profile = df_clean[FEATURES].groupby(clusters).mean().round(2)
profile['tracks'] = np.bincount(clusters)
profile['top_genres'] = (
    df_clean.groupby(clusters)['track_genre']
    .agg(lambda s: ', '.join(s.value_counts().head(4).index))
)
profile.index.name = 'cluster'
display(profile)

The 7 clusters map to recognizable sound profiles (cluster numbers depend on the random
initialization, so they are described by their dominant features):

| Profile | Dominant features | Typical genres |
|---|---|---|
| Soft acoustic songs | high acousticness (~0.7), low energy (~0.4) | tango, romance, acoustic |
| Loud and intense | highest energy (~0.8) and loudness (~−5 dB) | heavy metal, hardstyle |
| Spoken word | very high speechiness (~0.85) | comedy, show tunes, kids |
| Energetic instrumental | high instrumentalness (~0.8) and energy (~0.7) | techno, study |
| Calm instrumental | lowest energy (~0.2) and loudness (~−21 dB) | new age, sleep, ambient |
| Upbeat and danceable | highest danceability and valence (~0.7) | salsa, dancehall, disco |
| Live recordings | very high liveness (~0.75) | pagode, sertanejo, samba |

Most clusters are defined by one or two dominant features (speech, instruments, live audience,
loudness). The largest one, "upbeat and danceable", is the closest to a *mood*.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Clusters projected on the first two principal components
plt.figure(figsize=(10,6))
sns.scatterplot(
    x='PC1', y='PC2',
    hue='cluster',
    data=df_pca,
    palette='tab10',
    alpha=0.6
)
plt.title('Track clusters (K-Means + PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='Cluster')
plt.show()

## **3D PCA**

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd

# PCA with 3 components
pca_3d = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(df_scaled)  # df_scaled is the standardized dataset

df_pca_3d = pd.DataFrame(data=X_pca_3d, columns=['PC1', 'PC2', 'PC3'])
df_pca_3d['cluster'] = clusters  # K-Means clusters

df_pca_3d.head()

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# One color per cluster
colors = ['#5DF534', '#425B26', '#151716', '#876E03', '#E8CECF',
          '#023411', '#042E54', '#2572D3', '#FD0D0C', '#169C90']
df_pca_3d['color'] = df_pca_3d['cluster'].map({i: colors[i] for i in range(n_clusters)})

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter3D(df_pca_3d['PC1'], df_pca_3d['PC2'], df_pca_3d['PC3'], c=df_pca_3d['color'], alpha=0.6)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('Track clusters (K-Means + 3D PCA)')

plt.show()

## **Interactive 3D PCA (Plotly)**

In [ ]:
import plotly.express as px

# Add track info for the hover tooltip
df_pca_3d['track_name'] = df_clean['track_name'].values
df_pca_3d['artists'] = df_clean['artists'].values
df_pca_3d['track_genre'] = df_clean['track_genre'].values

fig = px.scatter_3d(
    df_pca_3d,
    x='PC1',
    y='PC2',
    z='PC3',
    color='cluster',
    opacity=0.7,
    hover_data=['track_name', 'artists', 'track_genre'],
    title='Track clusters (K-Means + interactive 3D PCA)'
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(scene=dict(
    xaxis_title='PC1',
    yaxis_title='PC2',
    zaxis_title='PC3'
))

fig.show()

## **Observations about the clusters**

Looking at the clusters produced by K-Means:

* The algorithm **groups tracks by acoustic, numeric features** such as *danceability*, *energy*, *loudness*, *tempo*, *valence*, *instrumentalness*, etc.
* As a result, tracks in the same cluster share **similar sound profiles**: comparable rhythm, intensity and tonality.
* However, **these similarities do not necessarily reflect genre, theme or vibe**. Two tracks can be close in acoustic terms even if one is light indie pop and the other is country or melodic rock, with completely different lyrics.
* This happens because the recommendation relies **only on objective audio features**, which capture sound patterns but not context, emotion or narrative.

## **Recommending similar tracks (nearest neighbours)**

The first version of this notebook recommended tracks by:
1. keeping only tracks from the reference track's K-Means cluster, and
2. ranking them by Euclidean distance in **3D PCA space**.

Both steps lose information:

* **PCA 3D** squeezes 9 features into 3 components, so tracks that look close in the projection
  can be far apart in reality. This is how a black metal track ended up as a top match for
  "Brooklyn Baby" (see `baseline_results.md`).
* **Cluster restriction** adds nothing useful: the reference cluster has thousands of tracks, so
  they still have to be ranked by distance, and the cluster borders cut off real neighbours
  (measured below).

**New approach: k-nearest neighbours (kNN)** on the 9 standardized features. Each track is a point
in 9-dimensional space; the recommendations are simply the closest points. No clusters, no
projection, no training.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn_euclidean = NearestNeighbors(metric='euclidean').fit(df_scaled)

# How often do a track's 10 nearest neighbours fall in a different K-Means cluster?
# (n_neighbors=11 because the closest point to each track is the track itself)
_, neighbour_idx = nn_euclidean.kneighbors(df_scaled, n_neighbors=11)
crosses_border = clusters[neighbour_idx[:, 1:]] != clusters[:, None]

print(f"Top-10 neighbours that are in another cluster: {crosses_border.mean():.1%}")
print(f"Tracks with at least one top-10 neighbour in another cluster: {crosses_border.any(axis=1).mean():.1%}")

In [ ]:
def find_track(name, artist):
    """Row index of a track, matching the name exactly and one of its credited artists exactly.

    `artists` holds several names separated by ';'. Matching a substring instead
    (str.contains("Lana")) also matched "Micky Flanagan" in the first version of the notebook.
    """
    credited = df_clean['artists'].str.split(';').apply(lambda names: artist in names)
    matches = df_clean[(df_clean['track_name'] == name) & credited]
    if matches.empty:
        raise ValueError(f"'{name}' by {artist} not found")
    return matches.index[0]


nn_cosine = NearestNeighbors(metric='cosine').fit(df_scaled)


def recommend(track_idx, n=10, model=nn_euclidean, same_genre=False, pool=2000):
    """The n tracks closest to `track_idx` in standardized feature space.

    With same_genre=True, only tracks sharing at least one genre with the reference are kept,
    taken from the `pool` nearest neighbours, so fewer than n tracks may be returned.
    """
    k = pool if same_genre else n
    distances, indices = model.kneighbors(df_scaled.iloc[[track_idx]], n_neighbors=k + 1)
    result = (
        df_clean.loc[indices[0], ['track_name', 'artists', 'genres']]
        .assign(distance=distances[0])
        .drop(index=track_idx, errors='ignore')
    )
    if same_genre:
        ref_genres = set(df_clean.at[track_idx, 'genres'])
        result = result[result['genres'].apply(lambda genres: bool(ref_genres & set(genres)))]
    return result.head(n)


# Exact artist match vs the substring search used before
substring = df_clean['artists'].str.contains('Lana', case=False)
exact = df_clean['artists'].str.split(';').apply(lambda names: 'Lana Del Rey' in names)
print(f"Tracks matching the substring 'Lana': {substring.sum()}")
print(f"Tracks credited to Lana Del Rey: {exact.sum()}")
display(df_clean.loc[exact, ['track_name', 'artists', 'genres']])

In [ ]:
ref_idx = find_track('Brooklyn Baby', 'Lana Del Rey')
display(df_clean.loc[[ref_idx], ['track_name', 'artists', 'album_name', 'genres'] + FEATURES])

by_euclidean = recommend(ref_idx, model=nn_euclidean)
by_cosine = recommend(ref_idx, model=nn_cosine)

print("Euclidean distance:")
display(by_euclidean)
print("Cosine distance:")
display(by_cosine)
print(f"Tracks in both lists: {len(set(by_euclidean.index) & set(by_cosine.index))} of 10")

In [ ]:
# Same search, keeping only tracks that share a genre with the reference
print(f"Reference genres: {df_clean.at[ref_idx, 'genres']}")
display(recommend(ref_idx, same_genre=True))

### Findings

* **Euclidean vs cosine:** the two lists mostly overlap, so the choice of metric is not what
  limits quality. We use **Euclidean distance**: it compares how far apart two tracks are on
  every feature, and it also works for searching around any target point (e.g. a set of desired
  feature values), not only around an existing track. Cosine only compares directions, so a track
  slightly above average on every feature and one far above average would look identical.
* **Clusters:** a noticeable share of each track's nearest neighbours lives in another cluster
  (see the check above), so restricting recommendations to one cluster would discard good matches.
* **Genre filter:** sharing a genre helps a little ("Another Love" by Tom Odell appears), but the
  dataset's genre labels are coarse: "Brooklyn Baby" is only tagged `pop`, and this dataset's `pop`
  is dominated by Indian pop. Genre works better as an **optional filter** than as a default.

**The honest limitation:** compared to the first version, the results no longer come from a lossy
projection, but they still do not match what a listener would call the same *vibe*. Dubstep, emo or
show tunes can sound similar to "Brooklyn Baby" on these 9 numbers (moderate energy, half
acoustic, very low valence) and still feel nothing like it. Audio features describe *how a track sounds*, not its
style, era, lyrics or cultural context. Streaming services fill that gap with listening behaviour
("people who play X also play Y"), which is not available here (Spotify closed its recommendations
API to new apps in 2024).

Getting closer to a listener's sense of similarity requires context beyond audio features, such
as listener-provided tags or listening behaviour.

### Before vs after: how close were the old recommendations really?

The first version of this notebook recommended the tracks below for "Brooklyn Baby"
(see `baseline_results.md`). Here we measure their **real** distance to the reference track in
the full 9-feature space, and their rank among all 81k tracks (1 = closest).

*Caveat:* the old results were produced with a different feature set (14 columns, including
`key`, `mode`, etc.), so this measures them against the current definition of similarity.

In [ ]:
old_recommendations = [
    # Version 1: PCA 3D distance, same cluster
    ("Let's Make Love", 'Faith Hill'),
    ('Alone in the Dark Mansion of Madness', 'Harley Poe'),
    ('Unspoken', 'Out of Nowhere'),
    ('Crying From The Bathroom', 'Koe Wetzel'),
    ('Ceremony for the Gathering of Death', 'Inquisition'),
    # Version 2: weighted PCA 3D distance, same cluster and same genre
    ('Another Love', 'Tom Odell'),
    ('The Hills', 'The Weeknd'),
    ('deja vu', 'Olivia Rodrigo'),
    ('Cherry', 'Lana Del Rey'),
]

ref_vector = df_scaled.iloc[ref_idx].to_numpy()
all_distances = np.linalg.norm(df_scaled.to_numpy() - ref_vector, axis=1)
# Rank 1 = closest track other than the reference itself
ranks = pd.Series(all_distances).rank(method='min').astype(int) - 1

rows = []
for name, artist in old_recommendations:
    idx = find_track(name, artist)
    rows.append({'track_name': name, 'artists': artist,
                 'distance': all_distances[idx], 'rank': ranks[idx]})

before = pd.DataFrame(rows)
display(before.round(2))

print(f"Median distance from 'Brooklyn Baby' to any track: {np.median(all_distances):.2f}")
print(f"Distance of the new top-10 (kNN): {by_euclidean['distance'].min():.2f} to {by_euclidean['distance'].max():.2f}")

**Result:** the old recommendations were not close neighbours at all. The black metal track
ranked *top 5* by the PCA 3D approach is around the median distance, i.e. about as similar
to "Brooklyn Baby" as a random track. Distances that look small in 3 PCA components can be large
in the full feature space.

The kNN results are, by construction, the closest tracks on every feature. They are more
principled and reproducible, but, as discussed above, being close on audio features is still not
the same as sharing a vibe.

## **Exporting the clean dataset**

Two files are written to `analysis/data/` (git-ignored, since the dataset must not be redistributed):

* **`tracks_clean.parquet`**: one row per unique track with its metadata, genres, the 9 raw audio
  features and `features_scaled`, the standardized 9-feature vector used for similarity.
* **`scaler.json`**: the feature order plus the mean and standard deviation used for standardization.
  Any new point (e.g. a set of target feature values) must be standardized with these same
  parameters to be comparable with `features_scaled`.

Parquet keeps column types (lists, booleans, floats) intact, unlike CSV.

In [ ]:
import json
from pathlib import Path

import sklearn

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

METADATA = ['track_id', 'track_name', 'artists', 'album_name', 'genres', 'popularity',
            'explicit', 'duration_ms', 'key', 'mode', 'time_signature']

tracks = df_clean[METADATA + FEATURES].copy()
# 'A;B' -> ['A', 'B']: credited artists as a list, like genres
tracks['artists'] = tracks['artists'].str.split(';')
tracks['features_scaled'] = list(df_scaled[FEATURES].to_numpy())

tracks.to_parquet(DATA_DIR / 'tracks_clean.parquet', index=False)

scaler_params = {
    'features': FEATURES,
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist(),
    'n_tracks': int(scaler.n_samples_seen_),
    'sklearn_version': sklearn.__version__,
}
(DATA_DIR / 'scaler.json').write_text(json.dumps(scaler_params, indent=2))

print(f"Exported {len(tracks):,} tracks with columns:")
print(tracks.columns.tolist())

In [ ]:
# Sanity checks on the exported files: read them back as a consumer would
exported = pd.read_parquet(DATA_DIR / 'tracks_clean.parquet')
params = json.loads((DATA_DIR / 'scaler.json').read_text())

vectors = np.stack(exported['features_scaled'].to_numpy())
raw = exported[params['features']].to_numpy()

assert len(exported) == len(df_clean)
assert exported['track_id'].is_unique
assert not exported[METADATA + FEATURES].isna().any().any()
assert vectors.shape == (len(exported), len(params['features']))
# Re-standardizing the raw features with scaler.json must reproduce the stored vectors
assert np.allclose((raw - params['mean']) / params['scale'], vectors)

print("All checks passed.")
display(exported.head(3))